In [170]:
# Libraries
import pandas as pd
import numpy as np
import xgboost as xgb
from scipy.stats import norm

In [171]:
# Config
DATA_FOLDER = "PW_0.5h"
TARGET = DATA_FOLDER
TRAIN_PATH = f"{DATA_FOLDER}/train.csv"
VAL_PATH = f"{DATA_FOLDER}/val.csv"
TEST_PATH = f"{DATA_FOLDER}/test.csv"

SENSORS = ["CMS air pressure", "Concentrazione ossigeno base", "Nitrogen pressure", "Oxygen over threshold"]
BINARY_SENSOR = "Oxygen over threshold"
WINDOWS = 20
STEP = 5
HORIZONS = (30, 60, 120)

In [172]:
# Load data
def load_data():
    train = pd.read_csv(TRAIN_PATH)
    val = pd.read_csv(VAL_PATH)
    test = pd.read_csv(TEST_PATH)
    binary_col = "Oxygen over threshold"
    for df in (train, val, test):
        df["timestamp"] = pd.to_datetime(df["timestamp"])

        df[binary_col] = np.where(
            np.isclose(df[binary_col], 22.437134398135598), 1, 0)
    return train, val, test

In [173]:
# Create survival targets
def create_survival_targets(data, target=TARGET):
    data = data.sort_values("timestamp").reset_index(drop=True).copy()
    changes = data[target].diff().fillna(0)
    failure_rows = changes == -1
    failure_times = data.loc[failure_rows, "timestamp"].sort_values().to_numpy()
    timestamps = data["timestamp"].to_numpy()
    next_idx = np.searchsorted(failure_times, timestamps)
    has_future_failure = next_idx < len(failure_times)
    data["event"] = has_future_failure.astype(int)
    data["next_failure_timestamp"] = pd.NaT
    data.loc[has_future_failure, "next_failure_timestamp"] = (failure_times[next_idx[has_future_failure]])
    data["time_to_event_min"] = np.nan
    data.loc[has_future_failure, "time_to_event_min"] = (
        data.loc[has_future_failure, "next_failure_timestamp"] - data.loc[has_future_failure, "timestamp"]
    ).dt.total_seconds() / 60
    ending = data["timestamp"].max()
    data.loc[~has_future_failure, "time_to_event_min"] = (ending - data.loc[~has_future_failure, 
        "timestamp"]).dt.total_seconds() / 60
    data["episode_id"] = failure_rows.cumsum().shift(fill_value=0).astype(int)
    return data[data["time_to_event_min"] > 0].copy()

In [174]:
# Extracteing sensor features
def extract_sensor_features(x, sensors=SENSORS):
    row = {}

    for sensor in sensors:
        values = x[sensor].to_numpy()

        if sensor == BINARY_SENSOR:
            row[f"{sensor}_current"] = values[-1]
            row[f"{sensor}_proportion"] = values.mean()
            row[f"{sensor}_transitions"] = np.sum(
                np.diff(values) != 0
            )

        else:
            row[f"{sensor}_current"] = values[-1]
            row[f"{sensor}_mean"] = values.mean()
            row[f"{sensor}_std"] = values.std()
            row[f"{sensor}_slope"] = np.polyfit(
                np.arange(len(values)), values, 1
            )[0]

    return row

In [175]:
# Create rolling features
def create_rolling_features(data, sensors=SENSORS, window=WINDOWS, step=STEP):
    rows = []
    for episode_id, episode in data.groupby("episode_id"):
        episode = episode.sort_values("timestamp").reset_index(drop=True)
        for end in range(window - 1, len(episode), step):
            x = episode.iloc[end - window + 1:end + 1]
            current = x.iloc[-1]
            row = {
                "episode_id": episode_id,
                "timestamp": current["timestamp"],
                "time_to_event_min": current["time_to_event_min"],
                "event": current["event"]
            }
            row.update(extract_sensor_features(x, sensors))
            rows.append(row)
    return pd.DataFrame(rows)

In [176]:
# Create labels and bounds
def create_cox_labels(data):
    times = data["time_to_event_min"].to_numpy(float)
    events = data["event"].to_numpy(int)
    return np.where(events == 1, times, -times)

def create_aft_bounds(data):
    times = data["time_to_event_min"].to_numpy(float)
    events = data["event"].to_numpy(int)
    low = times.copy()
    high = np.where(events == 1, times, np.inf)
    return low, high

In [177]:
# Concordance Index
def c_index(times, events, risk_scores):
    times = np.asarray(times, float)
    risk_scores = np.asarray(risk_scores, float)
    events = np.asarray(events).astype(bool)
    concordant = 0
    discordant = 0
    tied = 0
    for i in range(len(times)):
        if not events[i]:
            continue
        other = risk_scores[times > times[i]]
        if len(other) == 0:
            continue
        concordant += np.sum(risk_scores[i] > other)
        discordant += np.sum(risk_scores[i] < other)
        tied += np.sum(risk_scores[i] == other)
    total = concordant + discordant + tied
    if total == 0:
        return np.nan
    return (concordant + 0.5 * tied) / total

In [178]:
# IBS
def estimate_breslow_baseline(times, events, hazard_ratios):
    times = np.asarray(times, float)
    hazard_ratios = np.asarray(hazard_ratios, float)
    events = np.asarray(events).astype(bool)
    event_times = np.sort(np.unique(times[events]))
    inc = []
    for time in event_times:
        no_events = np.sum((times == time) & events)
        risk_sum = hazard_ratios[times >= time].sum()
        inc.append(no_events / risk_sum)
    return event_times, np.asarray(inc)

def cumulative_hazard_at_t(requested, times, cum_hazard):
    requested = np.asarray(requested, float)
    idx = np.searchsorted(times, requested, side="right") - 1
    out = np.zeros(len(requested))
    valid = idx >= 0
    out[valid] = cum_hazard[idx[valid]]
    return out

def estimate_censoring_survival(times, events):
    times = np.asarray(times, float)
    events = np.asarray(events, bool)
    censored = ~events
    unique_times = np.sort(np.unique(times))
    survival = 1.0
    values = []
    for time in unique_times:
        at_risk = np.sum(times >= time)
        censored_now = np.sum((times == time) & censored)
        if at_risk > 0:
            survival *= 1 - censored_now / at_risk
        values.append(survival)
    return unique_times, np.asarray(values)

def censoring_survival_at(requested, times, survival, left_limit=False):
    requested = np.asarray(requested, float)
    side = "left" if left_limit else "right"
    idx = np.searchsorted(times, requested, side=side) - 1
    out = np.ones(len(requested))
    valid = idx >= 0
    out[valid] = survival[idx[valid]]
    return out

def brier_scores_ipcw(censor_times, censor_survival, times, events, surv_probs, eval_times):
    times = np.asarray(times, float)
    events = np.asarray(events, bool)
    scores = []
    eps = 1e-12
    for i, time in enumerate(eval_times):
        predicted = surv_probs[:, i]
        g_time = max(censoring_survival_at([time], censor_times, censor_survival)[0], eps)
        fail = (times <= time) & events
        surv = times > time
        errors = np.zeros(len(times))
        if np.any(fail):
            g_fail = censoring_survival_at(times[fail], censor_times, censor_survival, left_limit=True)
            g_fail = np.maximum(g_fail, eps)
            errors[fail] = predicted[fail] ** 2 / g_fail
        errors[surv] = (1 - predicted[surv]) ** 2 / g_time
        scores.append(errors.mean())
    return np.asarray(scores)

def make_eval_grid(train_data, val_data, n=100):
    train_times = train_data["time_to_event_min"].to_numpy(float)
    val_times = val_data["time_to_event_min"].to_numpy(float)
    max_train = train_times.max()
    mask = val_times < max_train
    subset = val_data.loc[mask].copy()
    low = np.nextafter(subset["time_to_event_min"].min(), np.inf)
    high = np.nextafter(min(subset["time_to_event_min"].max(), max_train), -np.inf)
    eval_times = np.linspace(low, high, n)
    return subset, mask, eval_times

def ibs(scores, times):
    return np.trapezoid(scores, times) / (times[-1] - times[0])

In [179]:
# Training models
def train_models(X_train, X_val, roll_train, roll_val):
    dtrain_cox = xgb.DMatrix(X_train, label=create_cox_labels(roll_train))
    dval_cox = xgb.DMatrix(X_val, label=create_cox_labels(roll_val))
    cox_params = {
        "objective": "survival:cox",
        "eval_metric": "cox-nloglik",
        "eta": 0.05,
        "max_depth": 4,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "seed": 42
    }
    cox_model = xgb.train(params=cox_params, dtrain=dtrain_cox, num_boost_round=300, 
                          evals=[(dtrain_cox, "train"), (dval_cox, "validation")], early_stopping_rounds=20,
                         verbose_eval=False)
    train_low, train_high = create_aft_bounds(roll_train)
    val_low, val_high = create_aft_bounds(roll_val)
    dtrain_aft = xgb.DMatrix(X_train)
    dval_aft = xgb.DMatrix(X_val)
    dtrain_aft.set_float_info("label_lower_bound", train_low)
    dtrain_aft.set_float_info("label_upper_bound", train_high)
    dval_aft.set_float_info("label_lower_bound", val_low)
    dval_aft.set_float_info("label_upper_bound", val_high)
    aft_params = {
        "objective": "survival:aft",
        "eval_metric": "aft-nloglik",
        "aft_loss_distribution": "normal",
        "aft_loss_distribution_scale": 1.0,
        "eta": 0.05,
        "max_depth": 4,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "tree_method": "hist",
        "seed": 42,
    }
    aft_model = xgb.train(params=aft_params, dtrain=dtrain_aft, num_boost_round=300,
                          evals=[(dtrain_aft, "train"), (dval_aft, "validation")], early_stopping_rounds=20,
                          verbose_eval=False)
    return cox_model, aft_model, aft_params, dtrain_cox

In [180]:
# Comparing Cox and AFT
def compare_validation(cox_model, aft_model, aft_params, dtrain_cox, X_val, roll_train, roll_val):
    # Based on C-Index
    dval = xgb.DMatrix(X_val)
    cox_val_risk = cox_model.predict(dval, iteration_range=(0, cox_model.best_iteration + 1))
    cox_val_cindex = c_index(roll_val["time_to_event_min"], roll_val["event"], cox_val_risk)
    aft_val_risk = aft_model.predict(dval, iteration_range=(0, aft_model.best_iteration + 1))
    aft_val_cindex = c_index(roll_val["time_to_event_min"], roll_val["event"], -aft_val_risk)
    
    # Based on IBS
    cox_train_risk = cox_model.predict(dtrain_cox, iteration_range=(0, cox_model.best_iteration + 1))
    cox_event_times, cox_baseline_hazard = estimate_breslow_baseline(
        roll_train["time_to_event_min"], roll_train["event"], cox_train_risk)
    censor_times, censor_survival = estimate_censoring_survival(
        roll_train["time_to_event_min"], roll_train["event"])
    roll_val_ibs, val_mask, eval_times = make_eval_grid(roll_train, roll_val)
    cox_baseline_eval = cumulative_hazard_at_t(eval_times, cox_event_times, cox_baseline_hazard)
    cox_surv_probs = np.exp(-np.outer(cox_val_risk[val_mask], cox_baseline_eval))
    aft_val_margin = aft_model.predict(dval, output_margin=True, 
                                       iteration_range=(0, aft_model.best_iteration + 1))[val_mask]
    aft_scale = aft_params["aft_loss_distribution_scale"]
    z = (np.log(eval_times)[None, :] - aft_val_margin[:, None]) / aft_scale
    aft_surv_probs = norm.sf(z)
    val_times = roll_val_ibs["time_to_event_min"].to_numpy(float)
    val_events = roll_val_ibs["event"].to_numpy(bool)
    cox_bs = brier_scores_ipcw(censor_times, censor_survival, val_times, val_events, cox_surv_probs,
                               eval_times)
    aft_bs = brier_scores_ipcw(censor_times, censor_survival, val_times, val_events, aft_surv_probs,
                               eval_times)
    cox_val_ibs = ibs(cox_bs, eval_times)
    aft_val_ibs = ibs(aft_bs, eval_times)
    results = {
        "cox": {"val_cindex": cox_val_cindex, "val_ibs": cox_val_ibs},
        "aft": {"val_cindex": aft_val_cindex, "val_ibs": aft_val_ibs}
    }
    return results

In [181]:
# Testing 
def test_aft(aft_model, aft_params, X_test, roll_train, roll_test):
    dtest = xgb.DMatrix(X_test)
    aft_test_predict = aft_model.predict(dtest, iteration_range=(0, aft_model.best_iteration + 1))
    aft_test_cindex = c_index(roll_test["time_to_event_min"], roll_test["event"], -aft_test_predict)
    censor_times, censor_survival = estimate_censoring_survival(
        roll_train["time_to_event_min"], roll_train["event"])
    roll_test_ibs, test_mask, eval_times = make_eval_grid(roll_train, roll_test)
    aft_test_margin = aft_model.predict(dtest, output_margin=True, 
                                        iteration_range=(0, aft_model.best_iteration + 1))[test_mask]
    aft_scale = aft_params["aft_loss_distribution_scale"]
    z = (np.log(eval_times)[None, :] - aft_test_margin[:, None]) / aft_scale
    aft_surv_probs = norm.sf(z)
    test_times = roll_test_ibs["time_to_event_min"].to_numpy(float)
    test_events = roll_test_ibs["event"].to_numpy(bool)
    aft_bs = brier_scores_ipcw(censor_times, censor_survival, test_times, test_events, aft_surv_probs,
                               eval_times)
    aft_test_ibs = ibs(aft_bs, eval_times)
    return {"test_cindex": aft_test_cindex, "test_ibs": aft_test_ibs}
    

In [182]:
# Create live features
def create_life_features(data, sensors=SENSORS, window=WINDOWS):
    data = data.sort_values("timestamp").tail(window).copy()
    if len(data) < window:
        raise ValueError(f"Need {window} readings, but only received {len(data)}.")
    row = extract_sensor_features(data, sensors)
    return pd.DataFrame([row])

In [183]:
# Predict model
def predict_aft(model, feature_row, feature_columns, scale, horizons=HORIZONS):
    dmatrix = xgb.DMatrix(feature_row[feature_columns])
    margin = model.predict(dmatrix, output_margin=True,
                           iteration_range=(0, model.best_iteration + 1))[0]
    horizons = np.asarray(horizons, float)
    z = (np.log(horizons) - margin) / scale
    survival = norm.sf(z)
    failure = 1 - survival
    result = {"median_time_to_failure_min": float(np.exp(margin))}
    for horizon, surv_prob, fail_prob in zip(horizons, survival, failure):
        result[f"survive_{int(horizon)}_min"] = float(surv_prob)
        result[f"fail_within_{int(horizon)}_min"] = float(fail_prob)
    return result

In [184]:
# Create pipeline
def create_survival_pipeline(data, model, params, feature_columns):
    features = create_life_features(data)
    prediction = predict_aft(model, features, feature_columns, params["aft_loss_distribution_scale"])
    prediction["survival_model"] = "XGBoost AFT"
    prediction["timestamp"] = data["timestamp"].max()
    return prediction    

In [185]:
# Output to TrustGuard
def output_to_trustguard(pipeline_output):
    return {
        "timestamp": pipeline_output["timestamp"],
        "survival_model": pipeline_output["survival_model"],
        "failure_probability_30_min": (pipeline_output["fail_within_30_min"]),
        "failure_probability_60_min": (pipeline_output["fail_within_60_min"]),
        "failure_probability_120_min": (pipeline_output["fail_within_120_min"]),
        "median_time_to_failure_min": (pipeline_output["median_time_to_failure_min"]),
    }

In [186]:
# MAIN WORKFLOW

def main():
    # Loading data and feature engineering
    train, val, test = load_data()
    train_survival = create_survival_targets(train)
    val_survival = create_survival_targets(val)
    test_survival = create_survival_targets(test)
    roll_train = create_rolling_features(train_survival)
    roll_val = create_rolling_features(val_survival)
    roll_test = create_rolling_features(test_survival)
    metadata = ["episode_id", "timestamp", "time_to_event_min", "event"]
    feature_columns = [col for col in roll_train.columns if col not in metadata]
    X_train = roll_train[feature_columns]
    X_val = roll_val[feature_columns]
    X_test = roll_test[feature_columns]
    
    # Train the models
    cox_model, aft_model, aft_params, dtrain_cox = train_models(X_train, X_val, roll_train, roll_val)
    
    # Compare models
    validation_results = compare_validation(cox_model, aft_model, aft_params, dtrain_cox, X_val,
                                            roll_train, roll_val)
    print("\n------------- VALIDATION COMPARISON ----------------")
    print(f"COX: C-Index = {validation_results["cox"]["val_cindex"]:.4f}, "
          f"IBS = {validation_results["cox"]["val_ibs"]:.4f}")
    print(f"AFT: C-Index = {validation_results["aft"]["val_cindex"]:.4f}, "
          f"IBS = {validation_results["aft"]["val_ibs"]:.4f}")
    
    print("Selected survival model for test + live pipeline: XGBoost AFT")
    
    # Testing selected model (AFT)
    aft_test_result = test_aft(aft_model, aft_params, X_test, roll_train, roll_test)
    
    print("\n---------- TEST RESULTS -----------")
    print(f"AFT: C-Index = {aft_test_result["test_cindex"]:.4f}, "
          f"IBS = {aft_test_result["test_ibs"]:.4f}")
    example_timestamp = roll_test.iloc[0]["timestamp"]
    recent_sensor_data = test[test["timestamp"] <= example_timestamp].tail(WINDOWS).copy()
    pipeline_output = create_survival_pipeline(recent_sensor_data, aft_model, aft_params, feature_columns)
    trustguard_output = output_to_trustguard(pipeline_output)
    
    print("\n---------- OUTPUT TO TRUSTGUARD -------------")

    for key, value in trustguard_output.items():
        if isinstance(value, (float, np.floating)):
            print(key, ":", round(value, 4))
        else:
            print(key, ":", value)

    return {
        "cox_model": cox_model,
        "aft_model": aft_model,
        "validation_results": validation_results,
        "aft_test_results": aft_test_result,
        "feature_columns": feature_columns,
        "aft_params": aft_params,
        "trustguard_output": trustguard_output,
    }


if __name__ == "__main__":
    artifacts = main()


------------- VALIDATION COMPARISON ----------------
COX: C-Index = 0.7912, IBS = 0.6482
AFT: C-Index = 0.8055, IBS = 0.0901
Selected survival model for test + live pipeline: XGBoost AFT

---------- TEST RESULTS -----------
AFT: C-Index = 0.7675, IBS = 0.0798

---------- OUTPUT TO TRUSTGUARD -------------
timestamp : 2023-09-22 00:22:00
survival_model : XGBoost AFT
failure_probability_30_min : 0.0004
failure_probability_60_min : 0.0039
failure_probability_120_min : 0.0248
median_time_to_failure_min : 855.0558
